In [2]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os
from scipy.stats import gaussian_kde
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
from matplotlib import colors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)

TC_path1d = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/genesis-new-1d/'
TC_path2d = r'F:\study\data\noaa-20c\NOAA\NOAA_20C_TXT_30_all_bing\genesis-new-2d\\'
TCT_path = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/'
soda_path = r'F:\study\data\soda\\'
path_noaa = r'F:\study\data\noaa-20c\\'

In [3]:
def composite_ttest(da, event_mask, dim='time', equal_var=False):

    event_mask = xr.DataArray(event_mask, dims=[dim], coords={dim: da[dim]})
    
    # 分组
    da_event = da.where(event_mask, drop=True)
    da_clim  = da.where(~event_mask, drop=True)
    
    # 合成均值
    composite = da_event.mean(dim=dim).values
    
    # 各组统计量
    n1 = da_event.count(dim=dim)
    n2 = da_clim.count(dim=dim)
    m1 = da_event.mean(dim=dim)
    m2 = da_clim.mean(dim=dim)
    v1 = da_event.var(dim=dim, ddof=1)
    v2 = da_clim.var(dim=dim, ddof=1)
    
    # t 统计量（Welch）
    se = np.sqrt(v1 / n1 + v2 / n2)
    t_stat = (m1 - m2) / se
    
    # 自由度（Welch–Satterthwaite）
    if equal_var:
        df = n1 + n2 - 2
        # 合并方差
        sp2 = ((n1 - 1) * v1 + (n2 - 1) * v2) / df
        se = np.sqrt(sp2 * (1/n1 + 1/n2))
        t_stat = (m1 - m2) / se
    else:
        df = (v1/n1 + v2/n2)**2 / (
            (v1/n1)**2/(n1 - 1) + (v2/n2)**2/(n2 - 1)
        )
    
    # 双尾 p 值
    p_value = 2 * stats.t.sf(np.abs(t_stat), df)
    
    # 保持 xarray 结构
    # p_value = xr.DataArray(p_value, coords=t_stat.coords, dims=t_stat.dims)
    
    return composite, p_value

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2014-01')]
ep = ep[ep < np.datetime64('2014-01')]
mix = mix[mix < np.datetime64('2014-01')]
suc = suc[suc < np.datetime64('2014-01')]
events = [mix, cp, ep, suc]

tcg_grid = xr.open_dataset('./data-20c/NOAA-20C_TC_grid.nc').tcg_grid.loc['1900':'2015']
tct_grid = xr.open_dataset('./data-20c/NOAA-20C_TCT_grid.nc').tct_grid.loc['1900':'2015']
ssta = xr.open_dataset('./data/ssta.nc').sst.loc['1900':'2015']

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

tcg_grid -= tcg_grid.groupby('time.month').map(rolling)
tct_grid -= tct_grid.groupby('time.month').map(rolling)
ssta -= ssta.groupby('time.month').map(rolling)



In [7]:
ssta.coords['season'] = ('time', [1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1]*116)

In [18]:
ssta.groupby('season')[1][2:-1].coarsen(time=3).mean()

<xarray.DataArray 'sst' (time: 115, lat: 181, lon: 361)>
array([[[          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        ...,
        [          nan,  352.14273707,  329.96051407, ...,
          335.50606283,  341.05161667,           nan],
        [          nan,  127.54773458,  116.45662435, ...,
          113.68384806,  113.68384806,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan]],

       [[          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
...
        [          nan,  352.14273707,  329.96051407, ...,
          169.13939412,  174.68494542,           nan],
        [          nan,  127.54773458,  116.45662435, ...,
          -52.68282064,  -52.68282064,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan]],

       [[          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan],
        ...,
        [          nan, -479.69060008, -501.87282308, ...,
         -329.96060181, -324.41505432,           nan],
        [          nan, -371.5522639 , -382.64337158, ...,
         -385.41615295, -385.4161555 ,           nan],
        [          nan,           nan,           nan, ...,
                   nan,           nan,           nan]]])
Coordinates:
  * time     (time) datetime64[ns] 1901-01-01 1902-01-01 ... 2015-01-01
    month    (time) float64 5.0 5.0 5.0 5.0 5.0 5.0 ... 5.0 5.0 5.0 5.0 5.0 5.0
  * lat      (lat) int32 -90 -89 -88 -87 -86 -85 -84 ... 84 85 86 87 88 89 90
  * lon      (lon) int32 0 1 2 3 4 5 6 7 8 ... 353 354 355 356 357 358 359 360
    season   (time) float64 1.0 1.0 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0 1.0 1.0 1.0